# Лабораторна робота №2: класифікація та аналіз помилок моделі

**Варіант:** 7 — Titanic Dataset, ціль `Survived`.
Схема: дані → baseline → класифікатор → ймовірності → confusion matrix → FP/FN → threshold → ROC → висновок.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import cross_val_predict, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.titanic_workflow import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, RANDOM_STATE, load_titanic, make_cv,
    make_preprocessor, split_features_target, stratified_split,
)

WORK_DIR = Path.cwd().parent
FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Постановка задачі та початковий аналіз даних

Набір Titanic (Kaggle `train.csv`, 891 пасажир) описує пасажирів лайнера. **Об'єкт** — пасажир. **Вхідні ознаки:** клас квитка `Pclass`, стать `Sex`, вік `Age`, кількість братів/сестер/подружжя `SibSp` і батьків/дітей `Parch` на борту, вартість квитка `Fare`, порт посадки `Embarked`.

**Прикладна постановка:** оцінити ризик загибелі пасажира, щоб у сценарії евакуації пріоритезувати допомогу. Тому **позитивним класом** обрано подію, яку важливо виявити:

$y = 1$ — пасажир **не вижив** (`AtRisk = 1 − Survived`), $y = 0$ — вижив.

Пропуск пасажира з високим ризиком (FN) у такій постановці дорожчий, ніж зайва увага до того, хто врятувався б сам (FP).

In [2]:
df = load_titanic()
print("Розмір:", df.shape)
audit = pd.DataFrame({"dtype": df.dtypes.astype(str), "пропуски": df.isna().sum(),
                      "частка пропусків, %": df.isna().mean() * 100, "унікальних": df.nunique()})
display(audit.style.format({"частка пропусків, %": "{:.1f}"}))
print("Повних дублікатів рядків:", df.duplicated().sum())
X, y = split_features_target(df)
print("Рядків з однаковим набором 7 ознак:", X.duplicated().sum())
conflicts = pd.concat([X, y], axis=1).groupby(list(X.columns), dropna=False)["AtRisk"].nunique()
print("Груп однакових ознак із різними мітками:", (conflicts > 1).sum())

Розмір: (891, 13)


,dtype,пропуски,"частка пропусків, %",унікальних
PassengerId,int64,0,0.0,891
Survived,int64,0,0.0,2
Pclass,int64,0,0.0,3
Name,object,0,0.0,891
Sex,object,0,0.0,2
Age,float64,177,19.9,88
SibSp,int64,0,0.0,7
Parch,int64,0,0.0,7
Ticket,object,0,0.0,681
Fare,float64,0,0.0,248


Повних дублікатів рядків: 0
Рядків з однаковим набором 7 ознак: 127
Груп однакових ознак із різними мітками: 16


- `Age` має 177 пропусків (19,9%) — заповнюється медіаною train і додається індикатор пропуску (сам факт відсутності віку може бути інформативним).
- `Cabin` має 77% пропусків — не використовується. `PassengerId`, `Name`, `Ticket` — ідентифікатори/текст, що не узагальнюються, тому вилучені.
- `Embarked` має 2 пропуски — мода.
- Повних дублікатів немає, але 127 пасажирів мають такий самий вектор із 7 ознак, як хтось інший, і серед таких груп трапляються різні мітки — частина помилок неусувна за цих ознак.

In [3]:
counts = y.value_counts().sort_index()
shares = y.value_counts(normalize=True).sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Вижив (0)", "Не вижив (1)"], counts.values, color=["#3f7f93", "#c46a5a"])
for bar, c, s in zip(bars, counts.values, shares.values):
    ax.text(bar.get_x() + bar.get_width() / 2, c + 8, f"{c} ({s:.1%})", ha="center")
ax.set(title="Розподіл класів AtRisk", ylabel="Кількість пасажирів", ylim=(0, counts.max() * 1.15))
fig.tight_layout()
fig.savefig(FIGURES / "class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"P(y=0) = {shares[0]:.3f}, P(y=1) = {shares[1]:.3f}")

P(y=0) = 0.384, P(y=1) = 0.616


Дисбаланс помірний (61,6% / 38,4%). Важлива особливість: **позитивний клас — більшість**. Тому baseline «усі не виживуть» матиме Recall = 1 і F1 ≈ 0,76, хоча нічого не вміє. Через це, крім метрик позитивного класу, рахуємо **macro F1** — середнє F1 обох класів; саме воно показує, чи модель розрізняє класи.

## 2. Формування навчальної та тестової вибірок

In [4]:
X_train, X_test, y_train, y_test = stratified_split(X, y)
split_check = pd.DataFrame({
    "Вибірка": ["Весь набір", "Train", "Test"],
    "Об'єктів": [len(y), len(y_train), len(y_test)],
    "P(y=1)": [y.mean(), y_train.mean(), y_test.mean()],
})
display(split_check.style.format({"P(y=1)": "{:.3f}"}))
pd.Series(X_test.index, name="PassengerId").to_csv(RESULTS / "test_passenger_ids.csv", index=False)

,Вибірка,Об'єктів,P(y=1)
0,Весь набір,891,0.616
1,Train,712,0.617
2,Test,179,0.615


Стратифікація зберегла частку позитивного класу (0,617 / 0,616). Той самий поділ (`random_state=42`) використовується в ЛР3, ПР4 і ЛР4; ідентифікатори test збережено в `results/test_passenger_ids.csv`.

## 3. Preprocessing pipeline

| Група | Ознаки | Перетворення |
|---|---|---|
| Числові | `Age`, `SibSp`, `Parch`, `Fare` | `SimpleImputer(median, add_indicator=True)` → `StandardScaler` |
| Категоріальні | `Pclass`, `Sex`, `Embarked` | `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore")` |

`Pclass` кодується one-hot: три класи кают — категорії, а не рівновіддалена шкала. Увесь `ColumnTransformer` входить у `Pipeline`, тому медіани, середні і категорії обчислюються лише на train-частині кожного fold.

In [5]:
pre = make_preprocessor().fit(X_train)
print("Ознак після preprocessing:", len(pre.get_feature_names_out()))
print(list(pre.get_feature_names_out()))
age_median_train = pre.named_transformers_["numeric"].named_steps["imputer"].statistics_[0]
print(f"Медіана Age у train = {age_median_train}, у всьому наборі = {X['Age'].median()}")

Ознак після preprocessing: 13
['Age', 'SibSp', 'Parch', 'Fare', 'missingindicator_Age', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
Медіана Age у train = 29.0, у всьому наборі = 28.0


Медіана віку в imputer дорівнює медіані train (29,0), а не всього набору (28,0) — статистики тест-даних у preprocessing не потрапляють. Інші джерела витоку перевірено: `Survived` вилучено, ідентифікатори не використовуються.

## 4–5. Baseline і моделі класифікації: 5-fold stratified CV

- `DummyClassifier(most_frequent)` — baseline.
- `LogisticRegression` — лінійна ймовірнісна модель; масштабування всередині `Pipeline`.
- `GaussianNB` — коректний для перевірки, але його припущення (умовна незалежність ознак і нормальність кожної з них) порушуються: one-hot ознаки бінарні, `SibSp`/`Parch`/`Fare` сильно асиметричні, `Sex` і `Pclass` пов'язані з `Fare`. Тому очікуємо гірше калібровані ймовірності.

In [6]:
cv = make_cv()
scoring = {"Accuracy": "accuracy", "Precision": "precision", "Recall": "recall", "F1": "f1",
           "Macro F1": "f1_macro", "ROC-AUC": "roc_auc"}
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
}
rows, fold_rows = [], []
for name, model in models.items():
    scores = cross_validate(Pipeline([("preprocessor", make_preprocessor()), ("model", model)]),
                            X_train, y_train, cv=cv, scoring=scoring)
    row = {"Модель": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[metric] = values.mean()
        row[f"{metric} std"] = values.std(ddof=1)
    rows.append(row)
cv_results = pd.DataFrame(rows)
cv_results.to_csv(RESULTS / "cv_results.csv", index=False)
show = cv_results[["Модель"] + list(scoring)]
display(show.style.format({m: "{:.3f}" for m in scoring}))
display(cv_results[["Модель"] + [f"{m} std" for m in scoring]].style.format({f"{m} std": "{:.3f}" for m in scoring}))

,Модель,Accuracy,Precision,Recall,F1,Macro F1,ROC-AUC
0,Baseline,0.617,0.617,1.000,0.763,0.381,0.500
1,Logistic Regression,0.805,0.821,0.875,0.847,0.789,0.854
2,GaussianNB,0.775,0.824,0.809,0.816,0.764,0.839


,Модель,Accuracy std,Precision std,Recall std,F1 std,Macro F1 std,ROC-AUC std
0,Baseline,0.003,0.003,0.000,0.002,0.001,0.000
1,Logistic Regression,0.017,0.019,0.028,0.014,0.018,0.021
2,GaussianNB,0.021,0.020,0.023,0.018,0.021,0.042


- **Baseline**: Accuracy 0,617 = частка більшого класу, Recall 1,0 і F1 0,763 — але ROC-AUC 0,5 і macro F1 0,381. Високий F1 тут — артефакт того, що позитивний клас є більшістю; модель не розрізняє пасажирів.
- **Logistic Regression** перевищує baseline за всіма метриками, що враховують обидва класи: macro F1 0,789, ROC-AUC 0,854, при цьому Precision 0,821 і Recall 0,875.
- **GaussianNB** трохи гірший за LR за Accuracy, Recall, F1, macro F1 і ROC-AUC, Precision майже однаковий.
- Розкид між fold-ами в обох моделей невеликий (std macro F1 ≈ 0,02), тобто перевага LR не є наслідком одного вдалого fold-а.

## 6. Вибір моделі та прогноз на test

Обрано **Logistic Regression**: найкращі macro F1, F1, Recall і ROC-AUC при стабільних fold-ах, а також краще узгоджені з припущеннями ймовірності. Модель навчається на всій train і **один раз** прогнозує test.

In [7]:
final_model = Pipeline([("preprocessor", make_preprocessor()),
                        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]).fit(X_train, y_train)
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]
pd.DataFrame({"PassengerId": X_test.index, "y_true": y_test.values, "proba": y_proba, "y_pred": y_pred}).to_csv(
    RESULTS / "test_predictions.csv", index=False)

coef = pd.Series(final_model.named_steps["model"].coef_[0],
                 index=final_model.named_steps["preprocessor"].get_feature_names_out()).sort_values()
display(coef.to_frame("коефіцієнт (лог-шанси AtRisk)").style.format("{:+.3f}"))

,коефіцієнт (лог-шанси AtRisk)
Sex_female,-1.384
Pclass_1,-1.149
Embarked_Q,-0.214
Embarked_C,-0.202
Pclass_2,-0.165
Fare,-0.052
Parch,+0.086
missingindicator_Age,+0.108
Embarked_S,+0.296
SibSp,+0.337


Коефіцієнти узгоджуються з історією: чоловіча стать (+1,26) і 3-й клас (+1,20) найсильніше збільшують лог-шанси загибелі, жіноча стать (−1,38) і 1-й клас (−1,15) — зменшують; більший вік (+0,56 на 1 std) і більше `SibSp` (+0,34) теж підвищують ризик. Індикатор пропуску віку має невеликий додатний коефіцієнт (+0,11).

## 7. Confusion matrix

In [8]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Вижив (0)", "Не вижив (1)"]).plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set(title="Logistic Regression: confusion matrix (test, t = 0,5)", xlabel="Прогноз", ylabel="Факт")
fig.tight_layout()
fig.savefig(FIGURES / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

TN=51, FP=18, FN=16, TP=94


Для цієї задачі:

- **TP** — модель правильно позначила пасажира, який загинув, як «у групі ризику».
- **TN** — правильно визначено пасажира, який вижив.
- **FP** — пасажир вижив, а модель віднесла його до групи ризику: допомогу витрачено на того, хто врятувався б і без неї (втрачений ресурс екіпажу).
- **FN** — пасажир загинув, а модель вважала його безпечним: саме ця людина не отримала б пріоритетної допомоги.

**FN серйозніші**: їхня ціна — пропущене життя, тоді як ціна FP — витрачений, але відновлюваний ресурс.

## 8. Метрики на test set і порівняння з CV

In [9]:
test_row = {"Accuracy": accuracy_score(y_test, y_pred), "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred), "F1": f1_score(y_test, y_pred),
            "Macro F1": f1_score(y_test, y_pred, average="macro"), "ROC-AUC": roc_auc_score(y_test, y_proba)}
baseline_pred = np.ones_like(y_test)
base_row = {"Accuracy": accuracy_score(y_test, baseline_pred), "Precision": precision_score(y_test, baseline_pred),
            "Recall": recall_score(y_test, baseline_pred), "F1": f1_score(y_test, baseline_pred),
            "Macro F1": f1_score(y_test, baseline_pred, average="macro"), "ROC-AUC": 0.5}
cv_lr = cv_results.set_index("Модель").loc["Logistic Regression", list(scoring)]
compare = pd.DataFrame({"CV (LR)": cv_lr, "Test (LR)": pd.Series(test_row), "Test (baseline)": pd.Series(base_row)})
compare["Test − CV"] = compare["Test (LR)"] - compare["CV (LR)"]
compare.to_csv(RESULTS / "test_metrics.csv")
display(compare.style.format("{:.3f}"))

,CV (LR),Test (LR),Test (baseline),Test − CV
Accuracy,0.805,0.810,0.615,0.005
Precision,0.821,0.839,0.615,0.018
Recall,0.875,0.855,1.000,-0.020
F1,0.847,0.847,0.761,0.000
Macro F1,0.789,0.798,0.381,0.010
ROC-AUC,0.854,0.825,0.500,-0.030


На test Accuracy (0,810), F1 (0,847) і macro F1 (0,798) практично збігаються з CV; Precision трохи вища, Recall трохи нижчий (−0,02, менше за 1 std). Найбільше відхилення — ROC-AUC: 0,825 проти 0,854 у CV (≈ 1,5 std), що для test із 179 пасажирів є звичайним коливанням. Великого розриву немає, тож CV-оцінка надійна. Відносно baseline LR підвищила Accuracy з 0,615 до 0,810 і macro F1 з 0,381 до 0,798.

## 9. Аналіз помилково класифікованих об'єктів

In [10]:
analysis = df.set_index("PassengerId").loc[X_test.index, ["Name", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
analysis["actual"] = y_test
analysis["predicted"] = y_pred
analysis["probability"] = y_proba
analysis["error_type"] = np.select([(analysis.actual == 0) & (analysis.predicted == 1),
                                    (analysis.actual == 1) & (analysis.predicted == 0)], ["FP", "FN"], "")
errors = analysis[analysis.error_type != ""].copy()
errors["distance_to_threshold"] = (errors["probability"] - 0.5).abs()
errors = errors.sort_values(["error_type", "probability"])
errors.to_csv(RESULTS / "misclassified.csv")
print("Помилок:", len(errors), errors["error_type"].value_counts().to_dict())
display(errors[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "actual", "predicted", "probability",
                "error_type", "distance_to_threshold"]].style.format(
    {"Age": "{:.0f}", "Fare": "{:.1f}", "probability": "{:.3f}", "distance_to_threshold": "{:.3f}"}))

Помилок: 34 {'FP': 18, 'FN': 16}


,Pclass,Sex,Age,SibSp,Parch,Fare,actual,predicted,probability,error_type,distance_to_threshold
PassengerId,,,,,,,,,,,
298,1,female,2,1,2,151.6,1,0,0.032,FN,0.468
42,2,female,27,1,0,21.0,1,0,0.187,FN,0.313
313,2,female,26,1,1,26.0,1,0,0.197,FN,0.303
358,2,female,38,0,0,13.0,1,0,0.214,FN,0.286
374,1,male,22,0,0,135.6,1,0,0.281,FN,0.219
420,3,female,10,0,2,24.1,1,0,0.286,FN,0.214
405,3,female,20,0,0,8.7,1,0,0.333,FN,0.167
773,2,female,57,0,0,10.5,1,0,0.378,FN,0.122
658,3,female,32,1,1,15.5,1,0,0.427,FN,0.073


In [11]:
near = errors[errors["distance_to_threshold"] < 0.1]
confident = errors[errors["distance_to_threshold"] > 0.3]
print(f"Граничні помилки (|p − 0,5| < 0,1): {len(near)} із {len(errors)}")
print(f"Впевнені помилки (|p − 0,5| > 0,3): {len(confident)}")
display(confident[["Name", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "error_type", "probability"]].style.format(
    {"Age": "{:.0f}", "Fare": "{:.1f}", "probability": "{:.3f}"}))

Граничні помилки (|p − 0,5| < 0,1): 10 із 34
Впевнені помилки (|p − 0,5| > 0,3): 15


,Name,Pclass,Sex,Age,SibSp,Parch,Fare,error_type,probability
PassengerId,,,,,,,,,
298,"Allison, Miss. Helen Loraine",1,female,2,1,2,151.6,FN,0.032
42,"Turpin, Mrs. William John Robert (Dorothy Ann Wonnacott)",2,female,27,1,0,21.0,FN,0.187
313,"Lahtinen, Mrs. William (Anna Sylfven)",2,female,26,1,1,26.0,FN,0.197
289,"Hosono, Mr. Masabumi",2,male,42,0,0,13.0,FP,0.820
490,"Coutts, Master. Eden Leslie ""Neville""",3,male,9,1,1,15.9,FP,0.869
147,"Andersson, Mr. August Edvard (""Wennerstrom"")",3,male,27,0,0,7.8,FP,0.904
805,"Hedman, Mr. Oskar Arvid",3,male,27,0,0,7.0,FP,0.905
665,"Lindqvist, Mr. Eino William",3,male,20,1,0,7.9,FP,0.905
82,"Sheerlinck, Mr. Jan Baptist",3,male,29,0,0,9.5,FP,0.911


In [12]:
group = analysis.assign(error=analysis.error_type != "").groupby(["Sex", "Pclass"]).agg(
    n=("actual", "size"), actual_risk=("actual", "mean"), mean_proba=("probability", "mean"),
    errors=("error", "sum"), FP=("error_type", lambda s: (s == "FP").sum()), FN=("error_type", lambda s: (s == "FN").sum()))
group["error_rate"] = group["errors"] / group["n"]
group.to_csv(RESULTS / "errors_by_group.csv")
display(group.style.format({"actual_risk": "{:.2f}", "mean_proba": "{:.2f}", "error_rate": "{:.2f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(data=analysis, x="probability", hue="actual", bins=25, element="step", stat="count",
             palette={0: "#3f7f93", 1: "#c46a5a"}, ax=axes[0])
for _, r in errors.iterrows():
    axes[0].axvline(r["probability"], color="black", alpha=0.25, lw=1)
axes[0].axvline(0.5, color="black", ls="--")
axes[0].set(title="Ймовірності AtRisk на test (вертикальні лінії — помилки)", xlabel="P(y = 1 | x)", ylabel="Пасажирів")
axes[0].legend(["Не вижив (1)", "Вижив (0)"], title="Факт")
plot_group = group.reset_index()
plot_group["група"] = plot_group["Sex"].map({"male": "чол.", "female": "жін."}) + ", " + plot_group["Pclass"].astype(str) + " кл."
axes[1].bar(plot_group["група"], plot_group["FN"], color="#c46a5a", label="FN")
axes[1].bar(plot_group["група"], plot_group["FP"], bottom=plot_group["FN"], color="#3f7f93", label="FP")
axes[1].set(title="Помилки за статтю і класом (test)", ylabel="Кількість помилок")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / "error_analysis.png", dpi=160, bbox_inches="tight")
plt.show()

**Закономірності** (34 помилки: 18 FP, 16 FN; 10 з них граничні, 15 — впевнені):

- **FP — переважно чоловіки 3-го класу, які вижили** (11 з 18). Модель дає їм p ≈ 0,87–0,95, тобто це впевнені помилки: серед навчальних чоловіків 3-го класу загинуло ~85%, і за 7 ознаками «щасливчики» не відрізняються від решти. Серед FP є троє хлопчиків (`Master`, 3 і 9 років та один без віку) — діти рятувалися частіше, але лінійний ефект `Age` цього не передає.
- **FN — переважно жінки, які загинули** (11 з 16: шість із 3-го класу, четверо з 2-го, одна з 1-го). Найвпевненіша помилка — дворічна дівчинка 1-го класу (PassengerId 298, p = 0,032), яка за ознаками належить до найбезпечнішої групи.
- **Чоловіки 1-го класу**: 5 FN з p = 0,43–0,48, тобто біля порога. Фактичний ризик у цій групі на test 0,65, а середня ймовірність моделі лише 0,51: модель застосовує сильний «захисний» ефект `Pclass_1` однаково до чоловіків і жінок.

**Гіпотези щодо причин:**

1. *Ефект класу різний для статей.* Логістична регресія без взаємодій додає ефект `Pclass` однаково для чоловіків і жінок, тому недооцінює ризик чоловіків 1-го класу (FN біля порога) і переоцінює безпечність жінок 2–3 класу. Перевірка: додати взаємодію `Sex × Pclass` або нелінійну модель (дерево — ЛР4).
2. *Бракує ознаки «дитина/титул».* Хлопчики (`Master`) виживали значно частіше за дорослих чоловіків, але лінійний ефект `Age` не відтворює порогу «дитина». Перевірка: додати ознаку титулу з `Name` або порогову ознаку віку.
3. *Неусувний шум:* 16 груп однакових векторів ознак мають різні мітки, тобто частина результату визначалась факторами, яких немає в даних (розташування каюти, шлюпка, поведінка). Впевнені FP серед чоловіків 3-го класу — переважно такі випадки.

## 10. Дослідження decision threshold

In [13]:
def threshold_table(y_true, proba, thresholds):
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn_, fp_, fn_, tp_ = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        rows.append({"Threshold": t, "Precision": precision_score(y_true, pred, zero_division=0),
                     "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
                     "Macro F1": f1_score(y_true, pred, average="macro"), "FP": fp_, "FN": fn_})
    return pd.DataFrame(rows)


test_thresholds = threshold_table(y_test, y_proba, [0.3, 0.5, 0.7])
test_thresholds.to_csv(RESULTS / "threshold_test.csv", index=False)
display(test_thresholds.style.format({"Threshold": "{:.1f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}", "Macro F1": "{:.3f}"}))

,Threshold,Precision,Recall,F1,Macro F1,FP,FN
0,0.3,0.759,0.945,0.842,0.745,33,6
1,0.5,0.839,0.855,0.847,0.798,18,16
2,0.7,0.857,0.764,0.808,0.771,14,26


Щоб **вибрати** поріг без підгонки під test, використовуємо out-of-fold ймовірності train (кожен об'єкт прогнозує модель, що його не бачила) на сітці 0,05–0,95.

In [14]:
oof_proba = cross_val_predict(Pipeline([("preprocessor", make_preprocessor()),
                                        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
                              X_train, y_train, cv=cv, method="predict_proba")[:, 1]
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
oof_thresholds = threshold_table(y_train, oof_proba, grid)
oof_thresholds.to_csv(RESULTS / "threshold_oof.csv", index=False)
display(oof_thresholds[oof_thresholds["Threshold"].isin([0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7])].style.format(
    {"Threshold": "{:.2f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}", "Macro F1": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for metric, color in [("Precision", "#3f7f93"), ("Recall", "#c46a5a"), ("F1", "#6a8f3f"), ("Macro F1", "#8a6aa8")]:
    axes[0].plot(oof_thresholds["Threshold"], oof_thresholds[metric], label=metric, color=color)
    axes[0].scatter(test_thresholds["Threshold"], test_thresholds[metric], color=color, marker="s", s=40, edgecolor="black", zorder=3)
axes[0].axvline(0.4, color="gray", ls=":", label="обраний t = 0,4")
axes[0].set(title="Метрики від threshold (лінії — OOF train, квадрати — test)", xlabel="Threshold t", ylabel="Значення", ylim=(0, 1.02))
axes[0].legend(fontsize=8, loc="lower left")
axes[1].plot(oof_thresholds["Threshold"], oof_thresholds["FP"], label="FP", color="#3f7f93")
axes[1].plot(oof_thresholds["Threshold"], oof_thresholds["FN"], label="FN", color="#c46a5a")
axes[1].axvline(0.4, color="gray", ls=":")
axes[1].set(title="Кількість FP і FN від threshold (OOF train, n = 712)", xlabel="Threshold t", ylabel="Кількість")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / "threshold_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

,Threshold,Precision,Recall,F1,Macro F1,FP,FN
5,0.30,0.758,0.961,0.847,0.746,135,17
6,0.35,0.771,0.943,0.848,0.759,123,25
7,0.40,0.793,0.923,0.853,0.779,106,34
8,0.45,0.805,0.891,0.845,0.779,95,48
9,0.50,0.821,0.875,0.847,0.789,84,55
11,0.60,0.847,0.804,0.825,0.780,64,86
13,0.70,0.864,0.706,0.777,0.746,49,129


- Зменшення порога з 0,5 до 0,3 збільшує Recall і зменшує FN, але FP зростають. Підвищення до 0,7 робить позитивні прогнози точнішими, проте FN різко зростають.
- На test: при t = 0,3 FN 16 → 6, але FP 18 → 33; при t = 0,7 FP 18 → 14, а FN 16 → 26.
- На OOF train F1 позитивного класу майже рівний на інтервалі 0,3–0,5 і максимальний при t = 0,4 (0,853); macro F1 максимальний при 0,5 (0,789).
- **Пропонований поріг t = 0,4.** Оскільки FN (пропущений пасажир з високим ризиком) дорожчі за FP, поріг варто зсунути вниз від 0,5: на OOF перехід 0,5 → 0,4 зменшує FN з 55 до 34 ціною +22 FP, macro F1 знижується лише з 0,789 до 0,779. Подальше зниження до 0,3 дає ще −17 FN, але +29 FP і macro F1 0,746 — виграш уже не компенсує втрат. Точний вибір з урахуванням вартості помилок і ресурсних обмежень виконується у ПР4.

## 11. ROC-крива

In [15]:
fpr, tpr, roc_thr = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(fpr, tpr, color="#3f7f93", lw=2, label=f"Logistic Regression, ROC-AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Випадкове ранжування (TPR = FPR)")
for t, mk in [(0.3, "o"), (0.5, "s"), (0.7, "^")]:
    pred = (y_proba >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, pred).ravel()
    ax.scatter(fp_ / (fp_ + tn_), tp_ / (tp_ + fn_), marker=mk, s=60, zorder=3, label=f"t = {t}")
ax.set(title="ROC-крива (test)", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "roc_curve.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"ROC-AUC = {auc:.4f}")

ROC-AUC = 0.8246


ROC-AUC = 0,825 означає: для випадково взятих пасажира, що загинув, і пасажира, що вижив, модель приблизно у 82% випадків дає першому вищу ймовірність ризику. Це добре, але не ідеальне ранжування, що узгоджується з confusion matrix: помилки залишаються в обох напрямках, і жоден поріг не прибирає їх одночасно. Точки t = 0,3/0,5/0,7 показують, що зміна порога лише пересуває робочу точку вздовж тієї самої кривої.

## 12. Підсумковий аналіз

Див. `README.md` — там зібрано відповіді на підсумкові питання з фактичними числами.